In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver

In [3]:
# Define tools
@tool
def send_email(recipient: str, body: str) -> str:
    """Send an email to a user."""
    return f"Email successfully sent to {recipient}!"

@tool
def search_knowledge_base(query: str) -> str:
    """Search internal documentation."""
    return f"Results for: {query}"

# Memory checkpointer is REQUIRED to store state across pauses
checkpointer = MemorySaver()

In [5]:
# Build agent with HITL middleware on sensitive tools
agent = create_agent(
    model="gpt-4o",
    tools=[send_email, search_knowledge_base],
    checkpointer=checkpointer,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,  # Requires human approval before running send_email
                "search_knowledge_base": False,
            }
        )
    ]
)

In [7]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "session-124"}}

# Step 1: Initial invocation triggers email tool
events = agent.invoke(
    {"messages": [("user", "Send an email to boss@company.com saying I finished the report.")]},
    config=config
)

# Execution pauses here! Check state to inspect interrupted action:
state = agent.get_state(config)
print("Is state paused?", bool(state.next))
# Output: Is state paused? True

# Inspect what the human is being asked to approve
for action in events["__interrupt__"][0].value["action_requests"]:
    print(action["name"], action["args"])

# Step 2: Human reviews pending action and resumes execution
# Resume with one decision per interrupted tool call
resumed_events = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print(resumed_events["messages"][-1].content)

Is state paused? True
send_email {'recipient': 'boss@company.com', 'body': 'I finished the report.'}
The email has been successfully sent to boss@company.com with the message "I finished the report."
